# BBC News Politics Subcategory Classifier

This notebook implements the full subcategory classification pipeline for the **Politics** category of the BBC News dataset, following the same architecture used for the Business and Entertainment categories.

**Pipeline Overview:**
- Phase 1 — Data Ingestion
- Phase 2 — Text Cleaning
- Phase 3 — Vectorization (Jina AI)
- Phase 4 — UMAP + HDBSCAN Clustering
- Phase 5 — c-TF-IDF + KeyBERT Labeling
- Phase 5B — Verification Block
- Phase 6 — AI Subcategory Naming
- Phase 8 — Manual Noise Correction
- Phase 9 — Named Entity Recognition (NER) with April Events Extraction

## Project Overview
This notebook classifies **417 BBC Politics articles** into meaningful subcategories. Beyond subcategory classification, the pipeline also performs Named Entity Recognition (NER) to extract political figures and their roles, and identifies any events that took place or were scheduled to take place in April. using Jina Embeddings v5, UMAP dimensionality reduction, and HDBSCAN density-based clustering.

**Author:** BBC NLP Project 2
**Dataset:** 417 BBC Politics Articles
**Model:** Jina Embeddings v5 (text-small) with clustering adapter

---

## Phase 1: Data Ingestion & Structural Partitioning
In this phase we load all 417 articles from the `./politics` folder into a structured DataFrame. Each article is split into its title and body, and metadata is tracked alongside the raw text.

In [9]:
# ============================================================
# PHASE 1: DATA INGESTION & STRUCTURAL PARTITIONING
# ============================================================

import os
import pandas as pd

# 1. Define the path to the politics folder
folder_path = './politics'

# 2. Initialize our data collection list
all_articles = []

# 3. Loop through each .txt file in the politics folder
for filename in sorted(os.listdir(folder_path)):
    if filename.endswith('.txt'):
        file_path = os.path.join(folder_path, filename)

        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read().strip()

        # 4. Split the file into title (first line) and body (rest)
        lines = content.split('\n')
        title = lines[0].strip()
        body = '\n'.join(lines[1:]).strip()

        # 5. Track metadata alongside raw text
        all_articles.append({
            'file_name': filename,
            'article_id': int(filename.replace('.txt', '')),
            'title': title,
            'raw_text': body,
            'article_length': len(body.split())
        })

# 6. Load into a structured DataFrame
df = pd.DataFrame(all_articles)

# 7. Quick sanity check
print(f" Total articles loaded: {len(df)}")
print(f" Columns: {df.columns.tolist()}")
print(f" Average article length: {df['article_length'].mean():.0f} words")
print(f" Longest article: {df['article_length'].max()} words")
print(f" Shortest article: {df['article_length'].min()} words")
print(f"\n Sample:")
df.head()

 Total articles loaded: 417
 Columns: ['file_name', 'article_id', 'title', 'raw_text', 'article_length']
 Average article length: 449 words
 Longest article: 4428 words
 Shortest article: 84 words

 Sample:


,file_name,article_id,title,raw_text,article_length
0,001.txt,1,Labour plans maternity pay rise,Maternity pay for new mothers is to rise by £1...,445
1,002.txt,2,Watchdog probes e-mail deletions,The information commissioner says he is urgent...,375
2,003.txt,3,Hewitt decries 'career sexism',Plans to extend paid maternity leave beyond si...,519
3,004.txt,4,Labour chooses Manchester,The Labour Party will hold its 2006 autumn con...,249
4,005.txt,5,Brown ally rejects Budget spree,Chancellor Gordon Brown's closest ally has den...,499


## Phase 2: The Refinement Layer (Data Cleaning)

In this phase we clean all politics articles to prepare them for vectorization.
The cleaning process removes HTML tags and normalises whitespace to ensure
the text is consistent and ready for embedding.

In [7]:
# ============================================================
# PHASE 2: DATA CLEANING — POLITICS
# ============================================================
import re
import pandas as pd

def clean_article(text):

    # 1. HTML REMOVAL - confirmed present in raw BBC data
    text = re.sub(r'<[^>]+>', '', text)

    # 2. WHITESPACE CLEANUP
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()

    return text

df = pd.read_csv('politics_subcategories_output.csv')

if 'raw_text' not in df.columns:
    df = df.rename(columns={'cleaned_text': 'raw_text'})

df['cleaned_text'] = df['raw_text'].apply(clean_article)
df = df.drop(columns=['raw_text'])

df.to_csv('politics_subcategories_output.csv', index=False)

print(f"Total articles: {len(df)}")
print(f"\nPolitics cleaning complete!")

Total articles: 417

Politics cleaning complete!


## Phase 3: The Vectorization Layer (Jina-v5)

In this phase we convert all 417 politics articles into 1024-dimensional vector embeddings using the Jina AI API (`jina-embeddings-v5-text-small`, `text-matching` task). Embeddings are saved to `entertainment_embeddings.npy` to avoid repeated API calls.

In [11]:
# ============================================================
# PHASE 3: THE VECTORIZATION LAYER (JINA-V5) - RATE LIMIT FIX
# ============================================================

import requests
import numpy as np
import time
import os

JINA_API_KEY = "jina_0630e4e341144046ad7def8c563d0a7dcCRoxZqTtbVC15To_XeVA2-9PnKx"
JINA_URL = "https://api.jina.ai/v1/embeddings"

HEADERS = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {JINA_API_KEY}"
}

def get_embeddings(texts, batch_size=20):
    all_embeddings = []
    total_batches = (len(texts) + batch_size - 1) // batch_size

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        batch_num = (i // batch_size) + 1

        payload = {
            "model": "jina-embeddings-v5-text-small",
            "task": "text-matching",
            "normalized": True,
            "dimensions": 1024,
            "input": batch
        }

        success = False
        retries = 3

        while not success and retries > 0:
            try:
                response = requests.post(JINA_URL, headers=HEADERS, json=payload)
                response.raise_for_status()
                data = response.json()

                batch_embeddings = [item['embedding'] for item in data['data']]
                all_embeddings.extend(batch_embeddings)

                print(f" Batch {batch_num}/{total_batches} done — {len(all_embeddings)} articles embedded so far")
                success = True

                time.sleep(3)

            except Exception as e:
                retries -= 1
                print(f" Rate limit hit on batch {batch_num}, retrying in 10s... ({retries} retries left)")
                time.sleep(10)

        if not success:
            print(f" Failed on batch {batch_num} after all retries")
            break

    return np.array(all_embeddings)

# Only embed if embeddings don't already exist
if os.path.exists('politics_embeddings.npy'):
    print(" Embeddings already exist — skipping API call")
    embeddings = np.load('politics_embeddings.npy')
else:
    # Run on ALL 416 articles using title + content combined
    texts = (df['title'] + " " + df['cleaned_text']).tolist()
    print(f" Embedding {len(texts)} articles with Jina-v5 clustering adapter...\n")

    embeddings = get_embeddings(texts)

    # Save embeddings to disk permanently
    np.save('politics_embeddings.npy', embeddings)
    print(f"\n Embedding complete!")

print(f" Embedding matrix shape: {embeddings.shape}")
print(f"   → {embeddings.shape[0]} articles x {embeddings.shape[1]} dimensions")

 Embeddings already exist — skipping API call
 Embedding matrix shape: (417, 1024)
   → 417 articles x 1024 dimensions


## Phase 4: Dimensionality Reduction & Clustering

In this phase we reduce the 1024-dimensional embeddings to 3 dimensions using UMAP, then apply HDBSCAN to discover natural subcategory clusters within the 417 politics articles. Results are saved to disk to avoid recomputation.

In [12]:
# ============================================================
# PHASE 4: DIMENSIONALITY REDUCTION & CLUSTERING
# ============================================================

import umap
import hdbscan
import numpy as np
import os

# Load embeddings from disk if available
if os.path.exists('politics_embeddings.npy'):
    print(" Loading saved embeddings...")
    embeddings = np.load('politics_embeddings.npy')
    print(f" Embeddings shape: {embeddings.shape}")

# Check if we already have saved cluster results
if os.path.exists('politics_cluster_labels.npy') and os.path.exists('politics_reduced_embeddings.npy'):
    
    print(" Loading saved cluster results...")
    cluster_labels = np.load('politics_cluster_labels.npy')
    reduced_embeddings = np.load('politics_reduced_embeddings.npy')
    confidence_scores = np.load('politics_confidence_scores.npy')
    
    df['subcategory_id'] = cluster_labels
    df['confidence_score'] = confidence_scores

    n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
    n_noise = list(cluster_labels).count(-1)

    print(f" Subcategories found: {n_clusters}")
    print(f"  Noise articles: {n_noise}")
    print(f" Clustered articles: {len(cluster_labels) - n_noise}")

else:
    
    # 1. UMAP
    print(" Running UMAP dimensionality reduction...")
    reducer = umap.UMAP(
        n_components=3,
        n_neighbors=15,
        min_dist=0.0,
        metric='cosine',
        random_state=42
    )
    reduced_embeddings = reducer.fit_transform(embeddings)
    print(f" UMAP complete! Shape: {reduced_embeddings.shape}\n")

    # 2. HDBSCAN
    print(" Running HDBSCAN clustering...")
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=8,
        min_samples=4,
        metric='euclidean',
        cluster_selection_method='eom'
    )
    cluster_labels = clusterer.fit_predict(reduced_embeddings)
    confidence_scores = clusterer.probabilities_
    confidence_scores[cluster_labels == -1] = 0.0

    df['subcategory_id'] = cluster_labels
    df['confidence_score'] = confidence_scores

    # Save to disk permanently
    np.save('politics_cluster_labels.npy', cluster_labels)
    np.save('politics_reduced_embeddings.npy', reduced_embeddings)
    np.save('politics_confidence_scores.npy', confidence_scores)
    print(" Results saved to disk permanently!")

    n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
    n_noise = list(cluster_labels).count(-1)

    print(f" HDBSCAN complete!")
    print(f" Subcategories found: {n_clusters}")
    print(f"  Noise articles: {n_noise}")
    print(f" Clustered articles: {len(cluster_labels) - n_noise}")

# Cluster distribution
print(f"\n Articles per subcategory:")
print(df['subcategory_id'].value_counts().sort_index())

 Loading saved embeddings...
 Embeddings shape: (417, 1024)
 Loading saved cluster results...
 Subcategories found: 17
  Noise articles: 50
 Clustered articles: 367

 Articles per subcategory:
subcategory_id
-1     50
 0     16
 1     10
 2     13
 3     10
 4     18
 5     37
 6     23
 7     11
 8     12
 9     13
 10    45
 11    15
 12    58
 13     9
 14    24
 15    14
 16    39
Name: count, dtype: int64


## Phase 5: Thematic Labeling (c-TF-IDF)

In this phase we apply c-TF-IDF to extract the most distinctive keywords for each cluster. Words that appear frequently in one cluster but rarely in others are ranked highest, giving us meaningful thematic labels for each politics subcategory.

In [13]:
# ============================================================
# PHASE 5: THEMATIC LABELING (c-TF-IDF)
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import numpy as np

# 1. Separate clustered articles (exclude noise label -1)
clustered_df = df[df['subcategory_id'] != -1].copy()

# 2. Group all articles in each cluster into one big document
cluster_documents = clustered_df.groupby('subcategory_id')['cleaned_text'].apply(
    lambda texts: ' '.join(texts)
).reset_index()

# 3. c-TF-IDF — Compare word frequencies ACROSS clusters
# This finds words that are unique to each cluster (not just common everywhere)
vectorizer = TfidfVectorizer(
    max_features=10000,
    stop_words='english',
    ngram_range=(1, 2),    # Include bigrams like "government policy", "prime minister"
    min_df=1
)

tfidf_matrix = vectorizer.fit_transform(cluster_documents['cleaned_text'])
feature_names = vectorizer.get_feature_names_out()

# 4. Extract top keywords per cluster
def get_top_keywords(tfidf_matrix, feature_names, n=10):
    top_keywords = []
    for i in range(tfidf_matrix.shape[0]):
        row = tfidf_matrix[i].toarray()[0]
        top_indices = row.argsort()[-n:][::-1]
        keywords = [feature_names[idx] for idx in top_indices]
        top_keywords.append(keywords)
    return top_keywords

top_keywords = get_top_keywords(tfidf_matrix, feature_names, n=10)

# 5. Display keywords per cluster
print(" Top keywords per subcategory:\n")
for i, row in cluster_documents.iterrows():
    cluster_id = row['subcategory_id']
    keywords = top_keywords[i]
    print(f"Cluster {cluster_id:2d} | {', '.join(keywords)}")

# 6. Add top 3 keywords as semantic label to DataFrame
cluster_documents['semantic_label'] = [
    ' | '.join(kws[:3]) for kws in top_keywords
]

# 7. Map labels back to main DataFrame
label_map = dict(zip(cluster_documents['subcategory_id'], cluster_documents['semantic_label']))
df['semantic_label'] = df['subcategory_id'].map(label_map)
df['semantic_label'] = df['semantic_label'].fillna('Noise')

print(f"\n Semantic labels assigned!")
print(f"\n Cluster ID → Semantic Label:")
for cluster_id, label in sorted(label_map.items()):
    print(f"  Cluster {cluster_id:2d} → {label}")

 Top keywords per subcategory:

Cluster  0 | said, brown, aid, mr brown, africa, g8, mr, debt, countries, poverty
Cluster  1 | hunting, hunt, dogs, hunts, said, ban, countryside, hunting dogs, law, countryside alliance
Cluster  2 | workers, unions, regiments, pension, said, public sector, government, union, amicus, public
Cluster  3 | embargo, china, turkey, eu, said, arms, mr, straw, palestinian, lifted
Cluster  4 | mr, said, howard, party, mr howard, mcletchie, mr mcletchie, labour, election, tory
Cluster  5 | said, lord, mr blunkett, blunkett, mr, goldsmith, lord goldsmith, lord chancellor, lords, government
Cluster  6 | said, asylum, immigration, cards, id, mr, id cards, home, uk, people
Cluster  7 | jewish, livingstone, mr, said, mr livingstone, mayor, labour, posters, apologise, prince
Cluster  8 | ukip, kilroy, kilroy silk, silk, mr kilroy, party, mr, veritas, said, houston
Cluster  9 | drinking, said, binge, binge drinking, casinos, gambling, mcconnell, drunk, drink, games
Clus

## Phase 5B: KeyBERT + Cluster Verification

In this phase we extract keywords for each article using KeyBERT, then print sample titles and cleaned text snippets for each cluster. This allows us to read the actual content before assigning subcategory names to the 19 politics clusters.

In [14]:
# ============================================================
# PHASE 5B: KEYBERT + CLUSTER VERIFICATION
# ============================================================
from keybert import KeyBERT
import warnings
warnings.filterwarnings('ignore')

# 1. KeyBERT keyword extraction
print(" Loading KeyBERT model...")
kw_model = KeyBERT()
print(" KeyBERT model loaded!\n")

print(" Extracting keywords for each article...")
def extract_keywords(text, n=5):
    try:
        keywords = kw_model.extract_keywords(
            str(text),
            keyphrase_ngram_range=(1, 2),
            stop_words='english',
            top_n=n
        )
        return ', '.join([kw[0] for kw in keywords])
    except:
        return ''

df['article_keywords'] = df['cleaned_text'].apply(extract_keywords)
print(f" Keywords extracted for all {len(df)} politics articles!\n")

# 2. Cluster verification — read before naming
print("=" * 70)
print("      CLUSTER VERIFICATION — READ BEFORE NAMING")
print("=" * 70)

for cluster_id in sorted(df[df['subcategory_id'] != -1]['subcategory_id'].unique()):
    cluster_articles = df[df['subcategory_id'] == cluster_id]
    
    print(f"\n{'='*70}")
    print(f"CLUSTER {cluster_id} — {len(cluster_articles)} articles")
    print(f"{'='*70}")
    
    # Sample articles — focus on cleaned_text
    samples = cluster_articles.sample(min(3, len(cluster_articles)), random_state=42)
    for i, (_, row) in enumerate(samples.iterrows(), 1):
        print(f"\n Title {i}: {row['title']}")
        print(f" Keywords: {row['article_keywords']}")
        snippet = ' '.join(str(row['cleaned_text']).split()[:80])
        print(f" Text: {snippet}...")
    
    print(f"\n  WHAT LABEL SHOULD THIS CLUSTER GET?")
    print(f"{'='*70}")

 Loading KeyBERT model...
 KeyBERT model loaded!

 Extracting keywords for each article...
 Keywords extracted for all 417 politics articles!

      CLUSTER VERIFICATION — READ BEFORE NAMING

CLUSTER 0 — 16 articles

 Title 1: Tsunami debt deal to be announced
 Keywords: g8 britain, g8 ministers, ministers g8, countries gbp, finance ministers
 Text: Chancellor Gordon Brown has said he hopes to announce a deal to suspend debt interest repayments by tsunami-hit nations later on Friday. The agreement by the G8 group of wealthy nations would save affected countries GBP 3bn pounds a year, he said. The deal is thought to have been hammered out on Thursday night after Japan, one of the biggest creditor nations, finally signed up to it. Mr Brown first proposed the idea earlier this week. G8 ministers are also...

 Title 2: Brown visits slum on Africa trip
 Keywords: education africa, kenyan government, visited kenya, g8 aid, africa problems
 Text: Chancellor Gordon Brown has visited Kenya's bi

## Phase 6: Manual Label Assignment & Final Output

In this phase we assign professional subcategory names to each cluster based on the verification results from Phase 5B. The final output is saved to `politics_subcategories_output.csv`.

In [15]:
ai_label_map = {
    0:  "Africa Aid",
    1:  "Animal Welfare",
    2:  "Pension Strikes",
    3:  "Foreign Affairs",
    4:  "Conservative Party",
    5:  "Ministerial Scandals",
    6:  "Immigration Policy",
    7:  "Political Controversy",
    8:  "Fringe Parties",
    9:  "Public Health",
    10: "Counter-Terrorism",
    11: "Education Policy",
    12: "Labour Party",
    13: "Electoral Reform",
    14: "Liberal Democrats",
    15: "Public Spending",
    16: "Economic Policy",
    -1: "Noise"
}

mask_not_phase8 = ~df['semantic_label'].isin([])
df.loc[mask_not_phase8, 'semantic_label'] = df.loc[mask_not_phase8, 'subcategory_id'].map(ai_label_map)

df['confidence_score'] = confidence_scores
df.loc[df['subcategory_id'] == -1, 'confidence_score'] = 0.0
df.loc[df['semantic_label'] == 'Noise', 'confidence_score'] = 0.0

final_df = df[[
    'file_name',
    'article_id',
    'title',
    'cleaned_text',
    'subcategory_id',
    'semantic_label',
    'confidence_score',
    'article_keywords'
]].copy().reset_index(drop=True)

final_df.to_csv('politics_subcategories_output.csv', index=False)

print("=" * 55)
print("    POLITICS CLASSIFIER — FINAL RESULTS")
print("=" * 55)
print(f" Total Articles:        {len(final_df)}")
print(f" Subcategories Found:   {final_df[final_df['semantic_label'] != 'Noise']['semantic_label'].nunique()}")
print(f" Clustered Articles:    {len(final_df[final_df['semantic_label'] != 'Noise'])}")
print(f" Noise Articles:        {len(final_df[final_df['semantic_label'] == 'Noise'])}")
print(f" Avg Confidence:        {final_df[final_df['semantic_label'] != 'Noise']['confidence_score'].mean():.2f}")
print("=" * 55)
print(f"\n Label Distribution:")
print(final_df['semantic_label'].value_counts())
print(f"\n Saved to politics_subcategories_output.csv")

    POLITICS CLASSIFIER — FINAL RESULTS
 Total Articles:        417
 Subcategories Found:   17
 Clustered Articles:    367
 Noise Articles:        50
 Avg Confidence:        0.89

 Label Distribution:
semantic_label
Labour Party             58
Noise                    50
Counter-Terrorism        45
Economic Policy          39
Ministerial Scandals     37
Liberal Democrats        24
Immigration Policy       23
Conservative Party       18
Africa Aid               16
Education Policy         15
Public Spending          14
Public Health            13
Pension Strikes          13
Fringe Parties           12
Political Controversy    11
Foreign Affairs          10
Animal Welfare           10
Electoral Reform          9
Name: count, dtype: int64

 Saved to politics_subcategories_output.csv


## Phase 8: Manual Noise Correction

In this phase we review all 50 noise articles and manually assign them to the most appropriate subcategory based on their content.

In [16]:
noise_assignments = {
    3:   "Labour Party",
    8:   "Labour Party",
    143: "Labour Party",
    318: "Labour Party",
    196: "Labour Party",
    297: "Labour Party",
    306: "Labour Party",

    7:   "Conservative Party",
    351: "Conservative Party",
    371: "Conservative Party",

    23:  "Education Policy",

    33:  "Public Spending",
    38:  "Public Spending",
    167: "Public Spending",
    369: "Public Spending",
    146: "Public Spending",

    59:  "Electoral Reform",
    298: "Electoral Reform",
    322: "Electoral Reform",
    233: "Electoral Reform",

    62:  "Political Controversy",
    185: "Political Controversy",
    177: "Political Controversy",
    219: "Political Controversy",

    68:  "Public Health",
    98:  "Public Health",
    103: "Public Health",
    121: "Public Health",
    124: "Public Health",
    170: "Public Health",
    90:  "Public Health",

    82:  "Foreign Affairs",
    142: "Foreign Affairs",
    222: "Foreign Affairs",
    383: "Foreign Affairs",
    391: "Foreign Affairs",
    129: "Foreign Affairs",
    188: "Foreign Affairs",

    123: "Parliamentary Governance",
    310: "Parliamentary Governance",

    235: "Liberal Democrats",
    340: "Liberal Democrats",

    292: "Fringe Parties",
    304: "Fringe Parties",

    302: "Election Campaign",
    315: "Ministerial Scandals",
    326: "Counter-Terrorism",
    380: "Economic Policy",
}

for article_id, label in noise_assignments.items():
    mask = final_df['article_id'] == article_id
    final_df.loc[mask, 'semantic_label'] = label

final_df.to_csv('politics_subcategories_output.csv', index=False)

remaining_noise = len(final_df[final_df['semantic_label'] == 'Noise'])
print("=" * 55)
print("    NOISE CORRECTION — FINAL RESULTS")
print("=" * 55)
print(f" Manually assigned:     {len(noise_assignments)}")
print(f" Remaining noise:       {remaining_noise}")
print(f" Total classified:      {len(final_df[final_df['semantic_label'] != 'Noise'])}")
print(f"\n Final Label Distribution:")
print(final_df['semantic_label'].value_counts())
print("=" * 55)

    NOISE CORRECTION — FINAL RESULTS
 Manually assigned:     48
 Remaining noise:       2
 Total classified:      415

 Final Label Distribution:
semantic_label
Labour Party                65
Counter-Terrorism           46
Economic Policy             40
Ministerial Scandals        38
Liberal Democrats           26
Immigration Policy          23
Conservative Party          21
Public Health               20
Public Spending             19
Foreign Affairs             17
Africa Aid                  16
Education Policy            16
Political Controversy       15
Fringe Parties              14
Pension Strikes             13
Electoral Reform            13
Animal Welfare              10
Noise                        2
Parliamentary Governance     2
Election Campaign            1
Name: count, dtype: int64


# Politics Pipeline — Complete 

417 articles processed  with 0 remaining noise. 2 noise articles were manually labelled in Phase 8, followed by a Phase 8B cleanup merging. 

In [17]:
noise_assignments_fix = {
    39:  "Political Controversy",
    176: "Animal Welfare",
}

for article_id, label in noise_assignments_fix.items():
    mask = final_df['article_id'] == article_id
    final_df.loc[mask, 'semantic_label'] = label

final_df.to_csv('politics_subcategories_output.csv', index=False)

print(f"Remaining noise: {len(final_df[final_df['semantic_label'] == 'Noise'])}")
print(final_df['semantic_label'].value_counts())

Remaining noise: 0
semantic_label
Labour Party                65
Counter-Terrorism           46
Economic Policy             40
Ministerial Scandals        38
Liberal Democrats           26
Immigration Policy          23
Conservative Party          21
Public Health               20
Public Spending             19
Foreign Affairs             17
Education Policy            16
Africa Aid                  16
Political Controversy       16
Fringe Parties              14
Pension Strikes             13
Electoral Reform            13
Animal Welfare              11
Parliamentary Governance     2
Election Campaign            1
Name: count, dtype: int64


## Phase 9: Named Entity Recognition with April Events Extraction
In this phase we address the project's **Desired** requirement from the project brief:

> *"Identify documents and extract the named entities for media personalities, clearly identifying their jobs (e.g. Politicians, TV/Film Personalities, Musicians)"*

> *"Extract summaries of anything that took place or is/was scheduled to take place in April."*

We use spaCy to extract named entities from each article's combined `title` and `cleaned_text` for the **politics category** (416 articles). First we verify spaCy is installed and the English language model is available.

## Phase 9A: Named Entity Recognition — Persons, Organisations, Locations (Politics Category)
In this phase we extract named entities from each article's combined `title` and `cleaned_text` using spaCy's `en_core_web_lg` model across all 416 politics articles. Three entity types are extracted:

- **Named Persons** — validated through a false positive filter that removes political party names, month names, acronyms, and single-word misclassifications, ensuring only real two-word-minimum person names are retained
- **Named Organisations** — political parties, government bodies, institutions, and other organisations mentioned
- **Named Locations** — countries, cities, constituencies, and regions identified as GPE or LOC entities

Each valid person is also classified by role using a context window of 80 characters surrounding the entity, matching against keyword lists for roles including Politician, Business Executive, Central Banker, Expert/Analyst, TV/Film Personality, Musician, Sports Personality, Journalist, and Other. The politics classifier includes expanded Politician keywords such as `parliament`, `cabinet`, `constituency`, `shadow`, `opposition`, and `party leader`. Results: **406** articles with named persons, **412** with organisations, **362** with locations.

In [8]:
# ============================================================
# PHASE 9A: NER — PERSONS, ORGS, LOCATIONS (POLITICS CATEGORY)
# ============================================================
import spacy
import pandas as pd

nlp = spacy.load('en_core_web_lg')
df = pd.read_csv('politics_subcategories_output.csv')
print(f" Loaded {len(df)} politics articles")

# ============================================================
# FALSE POSITIVE FILTER
# ============================================================
FALSE_POSITIVES = {
    # Political terms misclassified as persons
    'labour', 'conservative', 'liberal', 'democrat',
    'republican', 'tory', 'parliament', 'commons',
    'lib dem',
    # Movie/TV titles
    'alexander', 'catwoman', 'rings', 'halo', 'titanic',
    'batman', 'superman', 'spiderman', 'matrix', 'avatar',
    'shrek', 'gladiator', 'braveheart', 'terminator',
    # Days/Months
    'monday', 'tuesday', 'wednesday', 'thursday', 'friday',
    'saturday', 'sunday', 'january', 'february', 'march',
    'april', 'may', 'june', 'july', 'august', 'september',
    'october', 'november', 'december',
    # Common misclassified words
    'lord', 'sir', 'mr', 'mrs', 'ms', 'dr', 'prof',
    'aol', 'sec', 'gm', 'ibm', 'bbc', 'itv',
    # Org/campaign names misclassified as persons
    'hacan clearskies', 'stanley leisure', "dubs' succession",
    'stefan pakeerah', 'warren leblanc',
    # Historical figures used as metaphor
    'adolf hitler', 'winston churchill', 'josef stalin',
}

# Exact semantic_label values from politics dataset
POLITICS_LABELS = {
    'uk politics', 'election coverage', 'foreign policy', 'parliament',
    'government policy', 'party politics', 'international relations',
    'defence policy', 'legal affairs', 'public services', 'immigration',
    'northern ireland', 'scotland', 'wales', 'european union',
    'terrorism', 'war conflict', 'human rights', 'social policy'
}

def is_valid_person(name):
    if len(name) < 2:
        return False
    if name.lower() in FALSE_POSITIVES:
        return False
    if name.isupper():
        return False
    if not any(c.isalpha() for c in name):
        return False
    tokens = name.strip().split()
    if len(tokens) < 2:
        return False
    # filter possessives
    if name.endswith("'s") or name.endswith("s'"):
        return False
    # filter honorific-only entries
    honorifics = {'mr', 'mrs', 'ms', 'dr', 'prof', 'dame', 'sir', 'lord',
                  'guardsman', 'private', 'corporal', 'sergeant', 'colonel'}
    if tokens[0].lower().rstrip('.') in honorifics and len(tokens) == 2:
        return False
    # filter rank + name combos like "Scots Guardsman Allan Hendry"
    military = {'guardsman', 'private', 'corporal', 'sergeant', 'colonel',
                'general', 'captain', 'major', 'lieutenant', 'admiral'}
    if any(t.lower() in military for t in tokens):
        return False
    # filter "Name of Organisation" patterns
    if ' of ' in name.lower():
        return False
    # filter single initials like "J Smith"
    if len(tokens[0]) == 1 and tokens[0].isupper():
        return False
    # filter number entries
    if tokens[-1].isdigit():
        return False
    # filter comma-separated pairs
    if ',' in name:
        return False
    # filter entries with dots like "S. Young"
    if len(tokens[0]) <= 2 and '.' in tokens[0]:
        return False
    return True

# ============================================================
# KNOWN PERSON ROLE LOOKUP
# ============================================================
KNOWN_ROLES = {
    # UK Politicians
    'Tony Blair':           'Politician',
    'Gordon Brown':         'Politician',
    'Charles Kennedy':      'Politician',
    'Margaret Thatcher':    'Politician',
    'Michael Howard':       'Politician',
    'Geoff Hoon':           'Politician',
    'John Reid':            'Politician',
    'Patricia Hewitt':      'Politician',
    'Charles Clarke':       'Politician',
    'Tessa Jowell':         'Politician',
    'Richard Caborn':       'Politician',
    'Hilary Benn':          'Politician',
    'Ruth Kelly':           'Politician',
    'Menzies Campbell':     'Politician',
    'Oliver Letwin':        'Politician',
    'Peter Hain':           'Politician',
    'Paul Burstow':         'Politician',
    'Ed Davey':             'Politician',
    'Paul Flynn':           'Politician',
    'Dai Lloyd':            'Politician',
    'Anne Borthwick':       'Politician',
    'Roger Knapman':        'Politician',
    'Mark Croucher':        'Politician',
    'Iqbal Sacranie':       'Politician',
    'Jorg Wojahn':          'Politician',
    'Mark Serwotka':        'Politician',
    'John Hannett':         'Politician',
    'Jeff Evans':           'Politician',
    'Hugh Lanning':         'Politician',
    # International Politicians
    'Recep Erdogan':        'Politician',
    'Yasser Arafat':        'Politician',
    'Mahmoud Abbas':        'Politician',
    'George Bush':          'Politician',
    # Business Executives
    'Paul Sykes':           'Business Executive',
    # Expert/Analysts
    'Oleg Gordievsky':      'Expert/Analyst',
    'David Bell':           'Expert/Analyst',
    'Irene Khan':           'Expert/Analyst',
    'John Hilary':          'Expert/Analyst',
    'Brad Adams':           'Expert/Analyst',
    'Roger Bennett':        'Expert/Analyst',
    'Sundar Katwala':       'Expert/Analyst',
    'David Redvers':        'Expert/Analyst',
    'John Holliday':        'Expert/Analyst',
    'Robert Thame':         'Expert/Analyst',
    'Hannah Ward':          'Expert/Analyst',
    'Nigel Yeo':            'Expert/Analyst',
    'Eileen Lee':           'Expert/Analyst',
    'Keith McDonald':       'Expert/Analyst',
    'Kalan Kawa Karim':     'Expert/Analyst',
    'Andy Richards':        'Expert/Analyst',
    'Elizabeth Winkfield':  'Expert/Analyst',
    'Stephen Douglas':      'Expert/Analyst',
    'Terry Telford':        'Expert/Analyst',
    'John Bourn':           'Expert/Analyst',
    # Lawyers
    'Anne Maguire':         'Lawyer',
    'John Ison':            'Lawyer',
    'Moazzam Begg':         'Lawyer',
    'Martin Mubanga':       'Lawyer',
    'Feroz Abbasi':         'Lawyer',
    'Richard Belmar':       'Lawyer',
}

# ============================================================
# PERSON ROLE CLASSIFIER
# ============================================================
def classify_role(person, context, semantic_label=''):
    # Check known persons first
    if person in KNOWN_ROLES:
        return KNOWN_ROLES[person]

    context = context.lower()
    semantic_label = semantic_label.lower().strip() if semantic_label else ''

    if any(w in context for w in [
        'president', 'prime minister', 'minister', 'senator', 'mp ',
        'chancellor', 'governor', 'secretary of state', 'politician',
        'parliament', 'elected', 'vote', 'constituency', 'opposition',
        'cabinet', 'treasury', 'labour party', 'conservative', 'democrat',
        'republican', 'shadow', 'party leader', 'tory', 'backbench',
        'downing street', 'whitehall', 'the commons', 'the lords',
        'foreign secretary', 'home secretary', 'attorney general',
        'solicitor general', 'chief whip', 'speaker', 'mep', 'councillor',
        'ambassador', 'diplomat', 'envoy', 'commissioner', 'presidency',
        'union leader', 'trade union', 'general secretary', 'party chair'
    ]):
        return 'Politician'
    elif any(w in context for w in [
        'central bank', 'federal reserve', 'monetary policy',
        'interest rate', 'inflation target', 'bank of england',
        'european central bank', 'ecb', 'imf', 'world bank',
        'rate cut', 'rate hike', 'basis points'
    ]):
        return 'Central Banker'
    elif any(w in context for w in [
        'lawyer', 'attorney', 'legal counsel', 'defence counsel',
        'defendant', 'prosecution', 'lawsuit', 'sued', 'trial',
        'verdict', 'indicted', 'charged', 'acquitted', 'solicitor',
        'barrister', 'judge', 'court', 'tribunal', 'magistrate', 'judiciary'
    ]):
        return 'Lawyer'
    elif any(w in context for w in [
        'actor', 'actress', 'director', 'film', 'movie', 'tv', 'television',
        'presenter', 'host', 'comedian', 'starred', 'cast', 'role', 'screen'
    ]):
        return 'TV/Film Personality'
    elif any(w in context for w in [
        'singer', 'musician', 'rapper', 'band', 'music', 'album',
        'song', 'artist', 'record', 'chart', 'tour', 'concert'
    ]):
        return 'Musician'
    elif any(w in context for w in [
        'ceo', 'chief executive', 'chairman', 'founder', 'executive',
        'managing director', 'mogul', 'tycoon', 'entrepreneur',
        'investor', 'billionaire', 'company', 'corporation',
        'chief financial', 'cfo', 'coo', 'board of directors'
    ]):
        return 'Business Executive'
    elif any(w in context for w in [
        'analyst', 'economist', 'researcher', 'professor', 'scientist',
        'expert', 'academic', 'lecturer', 'scholar', 'strategy',
        'forecast', 'think tank', 'institute', 'consultant',
        'adviser', 'advisor', 'specialist', 'director of'
    ]):
        return 'Expert/Analyst'
    elif any(w in context for w in [
        'coach', 'manager', 'player', 'footballer', 'athlete',
        'champion', 'scored', 'match', 'tournament', 'league',
        'olympic', 'medal', 'world record', 'championship'
    ]):
        return 'Sports Personality'
    elif any(w in context for w in [
        'journalist', 'reporter', 'editor', 'correspondent', 'critic',
        'columnist', 'broadcaster', 'commentator', 'pundit', 'anchor',
        'newspaper', 'magazine', 'media', 'press'
    ]):
        return 'Journalist'
    # Subcategory fallback — politics labels default to Politician
    elif semantic_label in POLITICS_LABELS:
        return 'Politician'
    else:
        return 'Other'

# ============================================================
# NER EXTRACTION FUNCTION
# ============================================================
def extract_persons_orgs_locations(title, cleaned_text, semantic_label=''):
    combined = f"{title}. {cleaned_text}"
    doc = nlp(combined[:100000])

    persons = []
    person_roles = []
    orgs = []
    locations = []

    seen_persons = set()
    seen_orgs = set()
    seen_locations = set()

    for ent in doc.ents:

        if ent.label_ == 'PERSON':
            if ent.text not in seen_persons and is_valid_person(ent.text):
                seen_persons.add(ent.text)
                start = max(0, ent.start_char - 300)
                end = min(len(combined), ent.end_char + 300)
                context = combined[start:end]
                role = classify_role(ent.text, context, semantic_label)
                persons.append(ent.text)
                person_roles.append(f"{ent.text}: {role}")

        elif ent.label_ == 'ORG':
            if ent.text not in seen_orgs:
                seen_orgs.add(ent.text)
                orgs.append(ent.text)

        elif ent.label_ in ['GPE', 'LOC']:
            if ent.text not in seen_locations:
                seen_locations.add(ent.text)
                locations.append(ent.text)

    return {
        'named_persons':       ' | '.join(persons),
        'named_organisations': ' | '.join(orgs),
        'named_locations':     ' | '.join(locations),
        'person_roles':        ' | '.join(person_roles),
    }

# ============================================================
# RUN NER ON ALL ARTICLES
# ============================================================
print(" Running NER (Persons, Orgs, Locations)...")

named_persons       = []
named_organisations = []
named_locations     = []
person_roles        = []

for i, row in df.iterrows():
    result = extract_persons_orgs_locations(
        str(row['title']),
        str(row['cleaned_text']),
        str(row.get('semantic_label', ''))
    )
    named_persons.append(result['named_persons'])
    named_organisations.append(result['named_organisations'])
    named_locations.append(result['named_locations'])
    person_roles.append(result['person_roles'])

    if (i + 1) % 50 == 0:
        print(f"   Processed {i + 1}/{len(df)} articles...")

df['named_persons']       = named_persons
df['named_organisations'] = named_organisations
df['named_locations']     = named_locations
df['person_roles']        = person_roles

print(f"\n Person/Org/Location NER complete!")
print(f" Articles with Persons:   {sum(1 for x in named_persons if x)}")
print(f" Articles with Orgs:      {sum(1 for x in named_organisations if x)}")
print(f" Articles with Locations: {sum(1 for x in named_locations if x)}")

# ============================================================
# AUDIT — check for remaining 'Other'
# ============================================================
other_count = df['person_roles'].str.contains('Other', na=False).sum()
print(f"\n Articles still containing 'Other': {other_count}")

other_rows = df[df['person_roles'].str.contains("Other", na=False)].copy()

def extract_others(person_roles_str):
    entries = person_roles_str.split(" | ")
    return [e.strip() for e in entries if e.strip().endswith(": Other")]

other_rows['other_persons'] = other_rows['person_roles'].apply(extract_others)

for _, row in other_rows.iterrows():
    print(f"Article {row['article_id']} | {row['title'][:60]}")
    for person in row['other_persons']:
        print(f"   --> {person}")
    print()

 Loaded 417 politics articles
 Running NER (Persons, Orgs, Locations)...
   Processed 50/417 articles...
   Processed 100/417 articles...
   Processed 150/417 articles...
   Processed 200/417 articles...
   Processed 250/417 articles...
   Processed 300/417 articles...
   Processed 350/417 articles...
   Processed 400/417 articles...

 Person/Org/Location NER complete!
 Articles with Persons:   405
 Articles with Orgs:      413
 Articles with Locations: 363

 Articles still containing 'Other': 0


## Phase 9B: April Events Extraction (Politics Category)
In this phase we extract April-related events from each article using spaCy's DATE entity recognition. For every article where a DATE entity containing "April" is detected in the combined `title` and `cleaned_text`, a 300-character contextual window surrounding the mention is captured as the event summary. This satisfies the project's Desired requirement of identifying events that took place or were scheduled to take place in April.

In [9]:
# ============================================================
# PHASE 9B: NER — APRIL EVENTS (POLITICS CATEGORY)
# ============================================================

def extract_april_events(title, cleaned_text):
    combined = f"{title}. {cleaned_text}"
    doc = nlp(combined[:100000])

    april_events = []

    for ent in doc.ents:
        if ent.label_ == 'DATE':
            if 'april' in ent.text.lower():
                start = max(0, ent.start_char - 100)
                end = min(len(combined), ent.end_char + 200)
                april_events.append(combined[start:end].strip())

    return ' | '.join(april_events) if april_events else ''

print(" Extracting April events...")

april_events = []
for i, row in df.iterrows():
    result = extract_april_events(str(row['title']), str(row['cleaned_text']))
    april_events.append(result)

    if (i + 1) % 50 == 0:
        print(f"   Processed {i + 1}/{len(df)} articles...")

df['april_events'] = april_events

print(f"\n April Events extraction complete!")
print(f"  Articles with April Events: {sum(1 for x in april_events if x)}")

 Extracting April events...
   Processed 50/417 articles...
   Processed 100/417 articles...
   Processed 150/417 articles...
   Processed 200/417 articles...
   Processed 250/417 articles...
   Processed 300/417 articles...
   Processed 350/417 articles...
   Processed 400/417 articles...

 April Events extraction complete!
  Articles with April Events: 18


## Phase 9C: Save Complete NER Output (Politics Category)
In this phase we consolidate all extracted NER columns — `named_persons`, `named_organisations`, `named_locations`, `person_roles`, and `april_events` — into the final dataframe and save to `politics_ner_output.csv`. A sample verification of the first article is printed to confirm all columns are correctly populated before proceeding to the next category.

In [11]:
# ============================================================
# PHASE 9C: SAVE COMPLETE NER OUTPUT (POLITICS CATEGORY)
# ============================================================

df['named_persons']       = named_persons
df['named_organisations'] = named_organisations
df['named_locations']     = named_locations
df['person_roles']        = person_roles
df['april_events']        = april_events

df.to_csv('politics_ner_output.csv', index=False)
print(" Saved to politics_ner_output.csv")
print(f" Columns: {df.columns.tolist()}")
print(f"\n Sample row 1:")
print(f" Persons:  {df['named_persons'].iloc[0]}")
print(f" Orgs:     {df['named_organisations'].iloc[0]}")
print(f" Location: {df['named_locations'].iloc[0]}")
print(f" Roles:    {df['person_roles'].iloc[0]}")
print(f"  April:    {df['april_events'].iloc[0]}")

print("\n" + "=" * 55)
print("       POLITICS NER — FINAL SUMMARY")
print("=" * 55)
print(f" Total Articles:             {len(df)}")
print(f" Articles with Persons:      {sum(1 for x in named_persons if x)}")
print(f" Articles with Orgs:         {sum(1 for x in named_organisations if x)}")
print(f" Articles with Locations:    {sum(1 for x in named_locations if x)}")
print(f"  Articles with April Events: {sum(1 for x in april_events if x)}")
print("=" * 55)

 Saved to politics_ner_output.csv
 Columns: ['file_name', 'article_id', 'title', 'subcategory_id', 'semantic_label', 'confidence_score', 'article_keywords', 'cleaned_text', 'named_persons', 'named_organisations', 'named_locations', 'person_roles', 'april_events']

 Sample row 1:
 Persons:  Patricia Hewitt | Gordon Brown | Tony Blair | Sandra Gidley | David Frost
 Orgs:     Labour | the Trade and Industry | GMTV | Parliament | State | the General Election | the British Chambers of Commerce
 Location: 
 Roles:    Patricia Hewitt: Politician | Gordon Brown: Politician | Tony Blair: Politician | Sandra Gidley: Politician | David Frost: Politician
  April:    

       POLITICS NER — FINAL SUMMARY
 Total Articles:             417
 Articles with Persons:      405
 Articles with Orgs:         413
 Articles with Locations:    363
  Articles with April Events: 18


In [21]:
import pandas as pd

business = pd.read_csv('business_subcategories_output.csv')
entertainment = pd.read_csv('entertainment_subcategories_output.csv')
politics = pd.read_csv('politics_subcategories_output.csv')
sport = pd.read_csv('sport_subcategories_output.csv')
tech = pd.read_csv('tech_subcategories_output.csv')

print(f"Business:      {len(business)}")
print(f"Entertainment: {len(entertainment)}")
print(f"Politics:      {len(politics)}")
print(f"Sport:         {len(sport)}")
print(f"Tech:          {len(tech)}")
print(f"Total:         {len(business) + len(entertainment) + len(politics) + len(sport) + len(tech)}")

Business:      510
Entertainment: 386
Politics:      417
Sport:         511
Tech:          401
Total:         2225
